In [9]:
%pip install confluent-kafka fastavro faker google-auth urllib3

In [10]:
import base64
import datetime
import json
import time
import urllib3
import google.auth

from google.auth.transport.urllib3 import Request


def encode(source):
    return (
        base64.urlsafe_b64encode(source.encode("utf-8"))
        .decode("utf-8")
        .rstrip("=")
    )


class TokenProvider:

    HEADER = json.dumps({
        "typ": "JWT",
        "alg": "GOOG_OAUTH2_TOKEN"
    })

    def __init__(self):
        self.credentials, self.project = google.auth.default()
        self.http_client = urllib3.PoolManager()

        print("ADC project:", self.project)
        print("Credentials loaded:", type(self.credentials).__name__)

    def get_credentials(self):
        if not self.credentials.valid:
            self.credentials.refresh(Request(self.http_client))

        return self.credentials

    def get_jwt(self, creds):
        token_data = {
            "exp": creds.expiry.replace(
                tzinfo=datetime.timezone.utc
            ).timestamp(),

            "iat": datetime.datetime.now(
                datetime.timezone.utc
            ).timestamp(),

            "iss": "Google",
            "sub": creds.service_account_email
        }

        return json.dumps(token_data)

    def get_token(self, args):
        creds = self.get_credentials()

        token = ".".join([
            encode(self.HEADER),
            encode(self.get_jwt(creds)),
            encode(creds.token)
        ])

        expiry_utc = creds.expiry.replace(
            tzinfo=datetime.timezone.utc
        )

        now_utc = datetime.datetime.now(
            datetime.timezone.utc
        )

        expiry_seconds = (
            expiry_utc - now_utc
        ).total_seconds()

        return token, time.time() + expiry_seconds

In [11]:
import fastavro
import io

avro_schema = {
    "type": "record",
    "name": "CardTransactionEvent",
    "namespace": "com.fintech.lakehouse.streaming",

    "fields": [
        {"name": "transaction_id", "type": "string"},
        {"name": "customer_id", "type": "string"},
        {"name": "card_type", "type": "string"},
        {"name": "timestamp", "type": "long"},
        {"name": "amount", "type": "double"},
        {"name": "currency", "type": "string", "default": "USD"},
        {"name": "merchant_name", "type": "string"},
        {"name": "merchant_category", "type": "string"},
        {"name": "location_country", "type": "string"},
        {"name": "is_international", "type": "boolean"},
        {"name": "device_fingerprint", "type": "string"},
        {"name": "fraud_risk_score", "type": "float"}
    ]
}

parsed_schema = fastavro.parse_schema(avro_schema)

In [12]:
BOOTSTRAP_SERVERS = (
    "bootstrap-m2fh0vmmtugrztbwci63yeoe6."
    "8d4c02221.us-central1.managedkafka.s.cloud.goog:9092"
)

TOPIC_NAME = "card-transactions-topic"

In [13]:
from confluent_kafka import Producer

token_provider = TokenProvider()

producer_config = {
    "bootstrap.servers": BOOTSTRAP_SERVERS,

    "security.protocol": "SASL_SSL",
    "sasl.mechanisms": "OAUTHBEARER",

    "oauth_cb": token_provider.get_token,

    "acks": "all",
    "enable.idempotence": True
}

producer = Producer(producer_config)

print("Kafka producer initialized.")

ADC project: fintech-lakehouse-dev
Credentials loaded: Credentials
Kafka producer initialized.


In [14]:
import time
import random
import uuid

from faker import Faker

fake = Faker()

# ==========================================
# 1. FIXED CUSTOMER POOL
# ==========================================

CUSTOMERS = [f"CUST-{i:04d}" for i in range(1000, 2000)]

# Fixed 10 high-risk customers
HIGH_RISK_CUSTOMERS = {
    "CUST-1005",
    "CUST-1023",
    "CUST-1147",
    "CUST-1234",
    "CUST-1356",
    "CUST-1420",
    "CUST-1567",
    "CUST-1678",
    "CUST-1832",
    "CUST-1990"
}


# ==========================================
# 2. TRANSACTION GENERATOR
# ==========================================

def generate_transaction():

    customer_id = random.choice(CUSTOMERS)

    is_high_risk = customer_id in HIGH_RISK_CUSTOMERS

    fraud_risk_score = (
        round(random.uniform(0.85, 1.00), 2)
        if is_high_risk
        else round(random.uniform(0.00, 0.20), 2)
    )

    record = {
        "transaction_id": str(uuid.uuid4()),
        "customer_id": customer_id,
        "card_type": random.choice([
            "Visa", "MasterCard", "RuPay", "Amex"
        ]),
        "timestamp": int(time.time() * 1000),
        "amount": round(random.uniform(10.0, 1500.0), 2),
        "currency": "INR",
        "merchant_name": fake.company(),
        "merchant_category": random.choice([
            "retail", "electronics", "travel",
            "dining", "grocery", "other"
        ]),
        "location_country": "IN",
        "is_international": False,
        "device_fingerprint": str(uuid.uuid4())[:8],
        "fraud_risk_score": fraud_risk_score
    }

    return record

In [15]:
def delivery_report(err, msg):

    if err is not None:
        print(f"Delivery failed: {err}")

    else:
        print(
            f"Delivered | "
            f"Topic: {msg.topic()} | "
            f"Partition: {msg.partition()} | "
            f"Offset: {msg.offset()}"
        )

In [16]:
print(f"Starting producer: {TOPIC_NAME}")

try:

    while True:

        record = generate_transaction()

        # Serialize Python dictionary into Avro binary
        bytes_writer = io.BytesIO()

        fastavro.schemaless_writer(
            bytes_writer,
            parsed_schema,
            record
        )

        avro_bytes = bytes_writer.getvalue()

        # Publish to Kafka
        producer.produce(
            topic=TOPIC_NAME,
            key=record["customer_id"].encode("utf-8"),
            value=avro_bytes,
            callback=delivery_report
        )

        # Serve delivery callbacks
        producer.poll(0)

        print(
            f"Produced: {record['transaction_id']} | "
            f"Customer: {record['customer_id']} | "
            f"Amount: {record['amount']}"
        )

        time.sleep(1)

except KeyboardInterrupt:

    print("Stopping producer...")

finally:

    remaining = producer.flush(timeout=30)

    print(f"Undelivered messages remaining: {remaining}")
    print("Producer stopped.")

Starting producer: card-transactions-topic
Produced: 39c0fda2-916d-42b0-8369-7f782d6d1e6e | Customer: CUST-1451 | Amount: 105.39
Delivered | Topic: card-transactions-topic | Partition: 2 | Offset: 1078
Produced: 72d8f396-aaa6-4b77-88b3-9b4fceeca0a3 | Customer: CUST-1628 | Amount: 1092.61
Delivered | Topic: card-transactions-topic | Partition: 0 | Offset: 1065
Produced: 7ea0de39-9d45-4a13-8318-bbbe07ce6984 | Customer: CUST-1436 | Amount: 463.28
Delivered | Topic: card-transactions-topic | Partition: 2 | Offset: 1079
Produced: 072a74a2-0490-4b66-8447-fb98d0d3ed83 | Customer: CUST-1152 | Amount: 1236.75
Delivered | Topic: card-transactions-topic | Partition: 1 | Offset: 1055
Produced: 455e8556-d844-4c1d-bee9-6e00ac2f7884 | Customer: CUST-1355 | Amount: 760.9
Delivered | Topic: card-transactions-topic | Partition: 2 | Offset: 1080
Produced: 584389b4-c372-4a15-890f-2fa3f199476c | Customer: CUST-1766 | Amount: 904.83
Delivered | Topic: card-transactions-topic | Partition: 0 | Offset: 1066
Pr